In [1]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install -q scipy
!pip install -q h5py
!pip install -q matplotlib
!pip install -q torch==1.13.1+cu117 torchvision==0.14.1+cu117 torchaudio==0.13.1 --extra-index-url https://download.pytorch.org/whl/cu117
!pip install -q transformers
!pip install -q fuzzy_match
!pip install -q nltk
!pip install -q rouge
!pip install -q diffusers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 GB 979.1 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.3/24.3 MB 60.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 41.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torchdata 0.7.0 requires torch==2.1.0, but you have torch 1.13.1+cu117 which is incompatible.
torchtext 0.16.0 requires torch==2.1.0, but you have torch 1.13.1+cu117 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 22.8 MB/s eta 0:00:00


In [1]:
%cd /content/drive/MyDrive/Research/FINAL/Code/MMMM/

/content/drive/MyDrive/Research/FINAL/Code/MMMM


In [2]:
import sys
sys.path.append("./models")
sys.path.append("./training")
sys.path.append("./testing")
sys.path.append("./utils")
sys.path.append("./ZuCo")
sys.path.append("./trainer")

import torch
import torch.nn as nn
import torch.optim as optim

torch.set_default_dtype(torch.float16)

from master_init import *
from DSG import *

In [3]:
config = {
    "device" : "cuda:0",
    "device_ids" : [0]
}
device = config["device"]
device_ids = config["device_ids"]

# model = INITIALIZE_MODEL(device=device, device_ids=device_ids).to(device)
model = INITIALIZE_MODEL(device=device, device_ids=device_ids).to(device, dtype=torch.float16)

Cannot initialize model with low cpu memory usage because `accelerate` was not found in the environment. Defaulting to `low_cpu_mem_usage=False`. It is strongly recommended to install `accelerate` for faster and less memory-intense model loading. You can do so with: 
```
pip install accelerate
```
.


In [4]:
dataset_dict = INITIALIZE_DATALOADERS(
    keys=["Brain2Image"],
    bsz=[1]
)

In [5]:
dsg_tasks = DSGTasks()
dsg_tasks.add_task(
    DSGTask(
        task_name="EEG-IMG-CLASSIFICATION",
        dataset_tag="Brain2Image",
        criterion=nn.CrossEntropyLoss(), # CE Loss for Tenary Sentiment
        optimizer=optim.Adam,
        learning_rate=5e-3,
        converge_lim=2,
        converge_threshold=0.005,
        div_threshold=0.01
        )
    )

## Code

In [6]:
import torch
import torch.nn as nn

torch.set_default_dtype(torch.float16)

dataloader = dataset_dict["Brain2Image"]
model = model
optimizer = optim.Adam(model.parameters(), lr=5e-4)
criterion = nn.MSELoss()
device = "cuda"
device_ids = None
staging_device = None

if staging_device == None:
    if device_ids == None:
        staging_device = "cuda"
    else:
        staging_device = f"cuda:{device_ids[0]}"

from diffusers import AutoencoderKL

vae = AutoencoderKL.from_pretrained("CompVis/stable-diffusion-v1-4", subfolder="vae").to(device, dtype=torch.float16)

Cannot initialize model with low cpu memory usage because `accelerate` was not found in the environment. Defaulting to `low_cpu_mem_usage=False`. It is strongly recommended to install `accelerate` for faster and less memory-intense model loading. You can do so with: 
```
pip install accelerate
```
.


In [7]:
# from PIL import Image
# from torchvision import transforms as tfms

# to_tensor_tfm = tfms.ToTensor()

# def pil_to_latent(input_im):
#   # Single image -> single latent in a batch (so size 1, 4, 64, 64)
#   with torch.no_grad():
#     latent = vae.encode(to_tensor_tfm(input_im).unsqueeze(0).to("cuda")*2-1) # Note scaling
#   return 0.18215 * latent.latent_dist.sample() # or .mean or .sample

# def latents_to_pil(latents):
#   # bath of latents -> list of images
#   latents = (1 / 0.18215) * latents
#   with torch.no_grad():
#     image = vae.decode(latents)[0]
#   image = (image / 2 + 0.5).clamp(0, 1)
#   image = image.detach().cpu().permute(0, 2, 3, 1).numpy()
#   images = (image * 255).round().astype("uint8")
#   pil_images = [Image.fromarray(image) for image in images]
#   return pil_images

In [8]:
# results = {}
# for phase in ['train', 'dev']:
#     if phase == 'train':
#         model.train()    # Set model to training mode
#     else:
#         model.eval()     # Set model to evaluate mode

#     running_loss = 0.0
#     tot_cnt = 0

#     # Iterate over data.
#     current_data = dataloader[phase].load_data()
#     while not current_data["reset"]:
#         eeg, labels = current_data["data"], current_data["labels"]
#         latents = [pil_to_latent(Image.open(f"./data/Brain2Image/imageNet_images/{label.split('_')[0]}/{label}.JPEG").convert("RGB")) for label in labels]
#         if not len(latents) == 1:
#             print("[WARNING] BSZ NOT 1. AUTOMATICALLY DROPPING ALL BUT FIRST IN BATCH.")
#         latents = latents[0]

#         noise = torch.randn_like(latents)
#         bsz = latents.shape[0]

#         timesteps = torch.randint(0, 30, (bsz,), device=latents.device)
#         timesteps = timesteps.long()

#         noisy_latents = latents + noise

#         optimizer.zero_grad()

#         args_dict = {
#             "input_data_batch" : eeg.to(device, dtype=torch.float16),
#             "pool_result" : True,
#             "noisy_latents" : noisy_latents.to(device, dtype=torch.float16),
#             "timesteps" : timesteps,
#             "train" : True
#         }
#         model_pred = model("EEG-IMG-DIFFUSION", args_dict)
#         loss = criterion(model_pred, noise)
#         loss.backward()
#         optimizer.step()

#         # # statistics
#         running_loss += loss.item() * bsz
#         tot_cnt += bsz

#         current_data = dataloader[phase].load_data()

#     epoch_loss = running_loss / tot_cnt
#     results[phase] = epoch_loss
#     print('{} Loss: {:.4f}'.format(phase, epoch_loss))

OutOfMemoryError: ignored

## Training Code

In [6]:
import torch
import torch.nn as nn

torch.set_default_dtype(torch.float16)

dataloader = dataset_dict["Brain2Image"]
model = model
optimizer = optim.Adam(model.parameters(), lr=5e-4)
criterion = nn.MSELoss()
device = "cuda"
device_ids = None
staging_device = None

if staging_device == None:
    if device_ids == None:
        staging_device = "cuda"
    else:
        staging_device = f"cuda:{device_ids[0]}"

from diffusers import AutoencoderKL

vae = AutoencoderKL.from_pretrained("CompVis/stable-diffusion-v1-4", subfolder="vae").to(device, dtype=torch.float16)

Cannot initialize model with low cpu memory usage because `accelerate` was not found in the environment. Defaulting to `low_cpu_mem_usage=False`. It is strongly recommended to install `accelerate` for faster and less memory-intense model loading. You can do so with: 
```
pip install accelerate
```
.


In [7]:
import EEG_IMG_DIFFUSION

In [8]:
num_epochs = 10
for epoch_num in range(num_epochs):
    args_dict = {
        "model" : model,
        "dataloader" : dataloader,
        "optimizer" : optimizer,
        "criterion" : criterion,
        "device" : device,
        "device_ids" : device_ids,
        "staging_device" : staging_device,
        "vae" : vae
    }
    results = EEG_IMG_DIFFUSION.train(args_dict)
    model = results["model"]
    print(i, results["train_loss"], results["dev_loss"])

OutOfMemoryError: ignored